# Publish Works

Publishes tonight's accepted build as `openalex.works.openalex_works`, in one commit (oxjob #2052).

Runs only after Guardrails succeeds: Guardrails checks `openalex.works.openalex_works_candidate` (the view CreateWorksEnriched's last cell defines: the finished `openalex_works_build` with each work's stored dates), and this task writes exactly that view. A night Guardrails rejects never reaches `openalex_works`: the table keeps the last accepted build, and every reader outside the End 2 End (the Authors job at 12:00 UTC, Sources, merged ids, Embed_Qwen3, ~20 jobs) keeps reading it until a repair passes Guardrails and this task runs.

Next: Optimize_Works_Enriched compacts the table (no data change); every task that reads `openalex_works` depends on it.

The statement is #2004's publish (CreateWorksEnriched's last cell from night 2026-11-06 to 2026-11-17), moved here. Normal nights take the fast path, one `INSERT OVERWRITE ... BY NAME` (~6 min); a night whose build schema differs from `openalex_works`, or a duplicate id in a date table, takes the old path (date MERGEs into the build + DEEP CLONE, ~20 min more). A failure here leaves `openalex_works` untouched (both paths commit once): repair this task.

Revert: `git revert` the #2052 commits (CreateWorksEnriched publishes again, before Guardrails).


In [ ]:
-- oxjob #2052: publish, after Guardrails accepted the candidate. One commit replaces openalex_works with the finished
-- build and serves each work's stored dates (#679, #1873). Moved here unchanged from CreateWorksEnriched's last cell
-- (oxjob #2004), except that the fast path reads openalex_works_candidate, the view Guardrails checked (its SELECT is the
-- one #2004's cell inlined), so what is published is what was checked.
-- Normal nights: one INSERT OVERWRITE ... BY NAME, the table written once. It runs only when it must give what the
-- old path gives: openalex_works has exactly the build's columns and types (typeof(struct(*)): names, order, nested
-- fields; an INSERT can't drop or retype a column) and each date table has one row per id (a duplicate would
-- duplicate works here; the old MERGE fails on it). Otherwise the old path: both date MERGEs into the build, then a
-- DEEP CLONE that takes the build's schema (~20 min more), with the same table properties as before #2004. Either way
-- the published rows equal the candidate's.
BEGIN
  DECLARE fast BOOLEAN;
  SET fast = coalesce(
        (SELECT typeof(struct(*)) FROM openalex.works.openalex_works LIMIT 1)
          = (SELECT typeof(struct(*)) FROM openalex.works.openalex_works_build LIMIT 1)
    AND (SELECT count(*) = count(DISTINCT id) FROM openalex.works.openalex_works_hash)
    AND (SELECT count(*) = count(DISTINCT id) FROM openalex.works.openalex_works_created_date), FALSE);
  IF fast THEN
    INSERT OVERWRITE openalex.works.openalex_works BY NAME
    SELECT * FROM openalex.works.openalex_works_candidate;
  ELSE
    -- Copy authoritative dates from the persistent hash table back into the
    -- column so every consumer keeps reading openalex_works.updated_date as
    -- normal (oxjob #679). Covers both this run's changes (stamped in the hash
    -- table by CreateWorksEnriched) and rows whose date the DEEP CLONE reverted to works_base
    -- values. NULL guard: a hash row seeded with a NULL date must never
    -- overwrite a real date.
    MERGE INTO openalex.works.openalex_works_build AS target
    USING openalex.works.openalex_works_hash AS source
    ON target.id = source.id
    WHEN MATCHED AND source.updated_date IS NOT NULL
      AND target.updated_date IS DISTINCT FROM source.updated_date THEN
      UPDATE SET target.updated_date = source.updated_date;
    -- oxjob #1873: serve the stored date. Changes only works whose derived date moved later (anchors deleted and
    -- re-inserted, oldest records gone) or whose stored date was corrected. NULL guard as in the #679 MERGE above.
    MERGE INTO openalex.works.openalex_works_build AS target
    USING openalex.works.openalex_works_created_date AS source
    ON target.id = source.id
    WHEN MATCHED AND source.created_date IS NOT NULL
      AND target.created_date IS DISTINCT FROM source.created_date THEN
      UPDATE SET target.created_date = source.created_date;
    CREATE OR REPLACE TABLE openalex.works.openalex_works
    DEEP CLONE openalex.works.openalex_works_build
    TBLPROPERTIES (
      'delta.dataSkippingNumIndexedCols' = 36,
      'delta.deletedFileRetentionDuration' = '14 days',
      'delta.logRetentionDuration' = '14 days'
    );
  END IF;
END